# finqa-lora: GPU job runner

Clones the repo at `REF` and runs `scripts/kaggle_run.sh JOB_NAME`, which installs the pinned
libraries, downloads the pinned FinQA data and runs `jobs/JOB_NAME.sh`. All logic is in the repo;
to run a different job, change `JOB_NAME` only. The run stops at the first error.

Outputs: `/kaggle/working/JOB_NAME/`. Settings: Accelerator **GPU T4 x2**, Internet **on**,
then **Save Version → Save & Run All (Commit)**.

In [ ]:
REPO = "https://github.com/PritRai75/finqa-lora.git"
REF = "main"            # a branch or a commit hash
JOB_NAME = "timing"     # a file in jobs/, without .sh
WORKDIR = "/tmp/finqa-lora"
OUT = f"/kaggle/working/{JOB_NAME}"

In [ ]:
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2, then save again."
major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert (major, minor) >= (2, 6), "This torch is too old for TRL 1.17 (needs 2.6+)."
!nvidia-smi

In [ ]:
# "!" commands never stop the notebook on failure, so each one writes its exit code to a file
# and Python raises if it is not 0.
from pathlib import Path
def check(step):
    code = int(Path("/tmp/exit_code").read_text().strip())
    if code != 0:
        raise RuntimeError(f"{step} failed with exit code {code}: see the output above")

!rm -rf {WORKDIR} && git clone --quiet {REPO} {WORKDIR} && cd {WORKDIR} && git checkout --quiet {REF}; echo $? > /tmp/exit_code
check("git clone")
!cd {WORKDIR} && bash scripts/kaggle_run.sh {JOB_NAME} {OUT}; echo $? > /tmp/exit_code
check(f"job {JOB_NAME}")

In [ ]:
p = Path(OUT) / "report.json"
print(p.read_text() if p.exists() else "no report.json: see the output above")